# Session 2: duplicates, epoch coverage, and steps

This analysis uses participant 1066528 from the development set.
We assume each label timestamp starts a 30-second epoch. The original classifier uses this indexing convention.
The export semantics remain unconfirmed. See `docs/session-02.md` for the fixed source revision.

One-second bin occupancy measures observation density. It does not measure physiological signal quality.
No validity threshold is selected here.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from sleep_classifier.analysis import analyze

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/explore.yaml").exists())
output = analyze(ROOT / "configs/explore.yaml")
summary = json.loads((output / "summary.json").read_text())
pd.DataFrame(summary["sensors"]).T

## Coverage over the labeled night

The first plot shows unique readings per epoch. The second shows occupied one-second bins.
The third includes gaps from each epoch boundary to its nearest observation.
Heart rate has fewer expected observations than motion. Their bin fractions are not directly comparable.
Empty epochs have a 30-second gap.


In [ ]:
display(Image(filename=str(output / "coverage.png")))

## Inspect the empty epochs

These tables contain no imputed values. An observation at the epoch end belongs to the following epoch.
Removing exact duplicates does not remove epochs. Conflicting readings stop this analysis for investigation.


In [ ]:
for sensor in ("heart_rate", "motion"):
    epochs = pd.read_csv(output / f"{sensor}-epochs.csv")
    print(sensor)
    display(epochs.loc[epochs.sample_count == 0].head(10))

## Step history

The shaded interval covers the 24 hours before PSG. Missing records do not indicate zero activity.
Each step count belongs to the interval ending at the next recorded timestamp.
The final bin has an unknown endpoint. The analysis does not allocate step counts across partial intervals.


In [ ]:
display(Image(filename=str(output / "steps.png")))
display(pd.read_csv(output / "step-bins.csv").tail())

## Next decision

Use this participant to validate the audit tools. Check cohort integrity before selecting the held-out participants.
Then compare coverage distributions on development participants before selecting sensor-specific thresholds.
Report prediction coverage when evaluating any policy that rejects epochs.
